# V5 infrastructure: GPU checks on Kaggle (2 GPUs)

Settings (right panel): **Accelerator = the 2-GPU accelerator**, **Add Input** → your `v5-infra-kaggle` Dataset. Then **Run all** (about 20–30 minutes).

Code comes from the attached Dataset (`v5-infra-kaggle.zip`), or, if none is attached and Internet is on,
from the `infrastructure` branch on GitHub. Results land in `/kaggle/working/gpu_results/` and in
`gpu_results.zip` (Output panel). No keys, tokens or personal data are used.

In [ ]:
# 1. find the code: attached Dataset first, else clone the branch
import glob, os, shutil, subprocess
WORK = "/kaggle/working/07-infrastructure"
shutil.rmtree(WORK, ignore_errors=True)
hits = glob.glob("/kaggle/input/**/07-infrastructure/README.md", recursive=True)
zips = glob.glob("/kaggle/input/**/v5-infra-kaggle.zip", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), WORK)   # /kaggle/input is read-only
    print("code from Dataset:", hits[0])
elif zips:
    shutil.unpack_archive(zips[0], "/kaggle/working")
    print("code from zip:", zips[0])
else:
    print("No Dataset attached: Add Input -> your v5-infra-kaggle Dataset, or turn Internet on. Trying GitHub...")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "infrastructure", "https://github.com/pradeep-kumar-gs/era-v5-lab.git", "/kaggle/working/repo"], check=True)
    shutil.copytree("/kaggle/working/repo/teams/07-infrastructure", WORK)
    print("code from GitHub branch infrastructure")
os.chdir(WORK)
print(sorted(os.listdir(".")))

In [ ]:
# 2. the machine: expect 2 x GPU
!nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.version.cuda, "gpus", torch.cuda.device_count())

In [ ]:
# 3. run every GPU check (health, all-reduce, NCCL timeout, checkpoint tests on CUDA, save speed)
!python support/kaggle/run_gpu_checks.py

In [ ]:
# 4. show the results and pack them for download
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/gpu_results/summary.md").read()))
shutil.make_archive("/kaggle/working/gpu_results", "zip", "/kaggle/working/gpu_results")
print("Download /kaggle/working/gpu_results.zip from the Output panel, unzip into support/kaggle/results/")